# StatsBomb Open Data — Scope Validation

Validates the selected competition/season pairs and confirms data availability.

**Selected scope:**
- 1. Bundesliga 2023/2024
- La Liga 2020/2021
- Ligue 1 2021/2022
- Ligue 1 2022/2023

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

RAW = ROOT / 'data' / 'raw' / 'statsbomb'
print('Project root:', ROOT)

## 1. Competitions Available

In [ ]:
comps = pd.read_json(RAW / 'competitions.json')
target_ids = [9, 11, 7]
our_comps = comps[comps['competition_id'].isin(target_ids)][['competition_id','competition_name','season_id','season_name']]
print(f'Total competitions in StatsBomb open data: {len(comps)}')
our_comps

## 2. Matches Manifest

In [ ]:
manifest = pd.read_csv(RAW / 'matches_manifest.csv')
print(f'Total matches in our scope: {len(manifest)}')
print()
manifest.groupby('competition_label').size().rename('matches').reset_index()

## 3. Events File Inventory

In [ ]:
events_dir = RAW / 'events'
event_files = list(events_dir.glob('events_*.json'))
print(f'Event files downloaded: {len(event_files)} / {len(manifest)}')

three60_dir = RAW / 'three_sixty'
three60_files = list(three60_dir.glob('three_sixty_*.json'))
print(f'Three-sixty files: {len(three60_files)}')

## 4. Sample Event Schema Inspection

In [ ]:
sample_file = event_files[0]
with open(sample_file) as f:
    sample = json.load(f)

print(f'Sample match: {sample_file.stem}')
print(f'Total events: {len(sample)}')
print(f'Event keys: {list(sample[0].keys())}')
print()

event_types = pd.Series([e['type']['name'] for e in sample]).value_counts()
print('Event type distribution:')
event_types

In [ ]:
# Verify required fields
required = ['possession', 'possession_team', 'team', 'type', 'timestamp', 'period', 'play_pattern', 'location']
for field in required:
    present = sum(1 for e in sample if field in e)
    print(f'  {field:20s}: present in {present}/{len(sample)} events')

## 5. Possession Count Per Match

In [ ]:
poss_counts = []
for f in event_files[:20]:  # sample 20 matches
    with open(f) as fp:
        evs = json.load(fp)
    n_poss = len(set(e['possession'] for e in evs if 'possession' in e))
    n_shot_poss = len(set(e['possession'] for e in evs if e.get('type', {}).get('name') == 'Shot'))
    poss_counts.append({'match': f.stem, 'possessions': n_poss, 'shot_possessions': n_shot_poss})

poss_df = pd.DataFrame(poss_counts)
print('Possessions per match (sample of 20):')
print(poss_df.describe().round(1))